# GEDI processor walkthrough

This notebook queries native GEDI shots and prepares them for integration with TLS, LVIS, and BIOMASS products. `GEDIProvider` handles remote access; `GEDIProcessor` handles validation, filtering, derived RH metrics, reprojection, sampling, and interface conversion.

Install the required extras with:

```bash
python -m pip install -e '.[gedi,examples]'
```

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

from tomo_canopy.processors import GEDIProcessor

## Query configuration

The example uses a small WGS84 region in southwestern Ghana and projects results into UTM zone 30N. Confirm the appropriate target CRS for your study site.

In [ ]:
BBOX = (-2.70, 5.10, -2.50, 5.30)  # west, south, east, north
START_TIME = "2019-01-01"
END_TIME = "2024-07-25"
TARGET_CRS = "EPSG:32630"
VARIABLES = ["agbd", "rh"]

region = gpd.GeoDataFrame(
    {"name": ["example_roi"]},
    geometry=[box(*BBOX)],
    crs="EPSG:4326",
)

## Query and validate GEDI shots

`open()` delegates the query to `GEDIProvider` and validates the returned shot schema. An empty query remains a valid dataset so optional GEDI coverage does not break a multimodal workflow.

In [ ]:
processor = GEDIProcessor()
shots = processor.open(
    region,
    variables=VARIABLES,
    start_time=START_TIME,
    end_time=END_TIME,
)
processor.validate(shots, require_crs=True)
shot_count = shots.sizes.get("shot_number", 0)
print(f"Queried shots: {shot_count:,}")
shots

## Filter measurements and derive RH98

A shot is retained when all requested variables contain at least one valid value. For profile variables, validity is evaluated across the profile dimension. `add_rh_metric()` selects the nearest available percentile label.

In [ ]:
processed = shots
if shot_count:
    processed = processor.filter_valid(
        processed, variables=["agbd", "rh"], how="any"
    )
    processed = processor.add_rh_metric(processed, percentile=98)

print(f"Valid shots: {processed.sizes.get('shot_number', 0):,}")
processed

## Reproject and inspect spatial bounds

GEDI coordinates arrive as WGS84 longitude and latitude. Reprojection creates metric `x` and `y` coordinates while preserving the original geographic coordinates.

In [ ]:
projected = None
bounds = None
if processed.sizes.get("shot_number", 0):
    projected = processor.reproject(processed, target_crs=TARGET_CRS)
    bounds = processor.bounds(projected)
    print(bounds)
else:
    print("No valid shots to reproject.")

## Local coordinates and reproducible sampling

Local metric coordinates are useful for joint visualization with TLS. Sampling limits display cost while retaining all variables for the selected shots.

In [ ]:
local = None
preview = None
if projected is not None:
    local = processor.to_local_coordinates(projected)
    preview = processor.sample_shots(local, count=5_000, seed=42)
    print(f"Preview shots: {preview.sizes['shot_number']:,}")
    print(
        "Global origin:",
        local.attrs["coordinate_origin_x"],
        local.attrs["coordinate_origin_y"],
    )

## Plot processed shots

The processor returns Xarray datasets, so variables can be plotted directly or passed to the visualization module later.

In [ ]:
if preview is not None and preview.sizes["shot_number"]:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    locations = axes[0].scatter(
        preview.x,
        preview.y,
        c=preview.agbd,
        s=10,
        cmap="viridis",
    )
    axes[0].set(
        xlabel="Local x [m]",
        ylabel="Local y [m]",
        title="GEDI AGBD shots",
    )
    fig.colorbar(locations, ax=axes[0], label="AGBD")

    axes[1].hist(preview.rh98.dropna("shot_number"), bins=30)
    axes[1].set(xlabel="RH98 [m]", ylabel="Shot count", title="RH98 distribution")
    plt.tight_layout()
    plt.show()
else:
    print("No processed GEDI shots are available to plot.")

## Create a shared-interface layer

The projected native shots can be wrapped as a `DataLayer`. They are not added directly to a `DataCube`: the future GEDI cube generator must first aggregate the irregular footprints onto its shared `GridSpec`.

In [ ]:
gedi_layer = None
if projected is not None:
    gedi_layer = processor.to_data_layer(projected, name="gedi_shots")
    print("Layer name:", gedi_layer.name)
    print("Modality:", gedi_layer.modality)
    print("Geometry:", gedi_layer.geometry)
    print("CRS:", gedi_layer.crs)